In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 9 ###

### cell 9 optimized
# Quick feature engineering

df['count'] = 1

# Extract the first country using GPU-accelerated string ops instead of apply
# Split on comma and grab the first element

df['first_country'] = df['country'].str.split(',').str.get(0)

# Shorten common country names in one replace call (GPU)
df['first_country'] = df['first_country'].replace({
    'United States': 'USA',
    'United Kingdom': 'UK',
    'South Korea': 'S. Korea'
})

# Map ratings to age groups with GPU replace
ratings_ages = {
    'TV-PG': 'Older Kids',
    'TV-MA': 'Adults',
    'TV-Y7-FV': 'Older Kids',
    'TV-Y7': 'Older Kids',
    'TV-14': 'Teens',
    'R': 'Adults',
    'TV-Y': 'Kids',
    'NR': 'Adults',
    'PG-13': 'Teens',
    'TV-G': 'Kids',
    'PG': 'Older Kids',
    'G': 'Kids',
    'UR': 'Adults',
    'NC-17': 'Adults'
}

df['target_ages'] = df['rating'].replace(ratings_ages)

# Clean up the genre strings and split into lists using GPU str methods
df['genre'] = (
    df['listed_in']
      .str.replace(' ,', ',')
      .str.replace(', ', ',')
      .str.split(',')
)

# If you need to inspect unique target_ages on the GPU:
# df['target_ages'].drop_duplicates()